# Dota 2: команды, игроки и составы с 2026 года

Обзор выбранных серий: BLAST Slam, ESL One, DreamLeague Season, Esports World Cup,
PGL Wallachia и The International. Семейства выбираются пользователем, это не проверенный
автоматический tier Liquipedia. Квалификации и DreamLeague Division 2 исключены.

Пятёрка определяется по Steam32 ID участников карты. Смена пятёрки — **наблюдаемая** смена,
включая возможного стендина; дата первого матча не является официальной датой трансфера.
Профили игроков и команд хранятся отдельно. До начала 2026 года данные могут использоваться
для определения границ состава. Статистика ниже ограничивается выбранным периодом.

По умолчанию читается `data/analysis.json`, иначе явно отмеченное синтетическое демо.
`SCOUT_DATA_PATH` и `SCOUT_ANALYSIS_START` позволяют выбрать файл и дату без правки кода.
Сбор данных не запускается при выполнении ноутбука.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dota_scout.analysis import load_bundle, analysis_frames, to_match
from dota_scout.forecast import valid_lineup

default_path = ROOT / "data/analysis.json"
DATA_PATH = os.getenv("SCOUT_DATA_PATH") or str(default_path if default_path.exists()
    else ROOT / "tests/fixtures/demo.json")
BUNDLE = load_bundle(DATA_PATH)
ANALYSIS_START = pd.Timestamp(os.getenv("SCOUT_ANALYSIS_START", "2026-01-01"), tz="UTC")
print("Источник:", DATA_PATH)
print("Синтетические данные:", bool(BUNDLE.get("synthetic")))
print("Дата экспорта:", BUNDLE.get("exported_at", "демо"))
print("Матчей в исходном наборе:", len(BUNDLE["matches"]))
if not BUNDLE["matches"]:
    raise ValueError("Выборка пуста. Сначала загрузите матчи.")
teams, players, drafts = analysis_frames(BUNDLE)
if teams.empty:
    raise ValueError("Нет матчей с известными team_id.")
report = teams[teams.started_at >= ANALYSIS_START].copy()
print("Период анализа:", ANALYSIS_START, "—", teams.started_at.max())


## Покрытие и качество
Одна карта имеет две командные строки. Нельзя считать их двумя независимыми матчами.


In [ ]:
coverage = pd.Series({
    "карт в периоде": report.match_id.nunique(),
    "команд": report.team_id.nunique(),
    "турниров": report.league_id.nunique(),
    "командных строк с известной пятёркой": int(report.roster_known.sum()),
    "командных строк всего": len(report),
    "карт с драфтом": report.loc[report.has_draft, "match_id"].nunique(),
})
display(coverage.to_frame("значение"))
display(report.groupby(["league_id", "league_name"], dropna=False).agg(
    maps=("match_id", "nunique"), first=("started_at", "min"), last=("started_at", "max")))
print("Не обещаем полное покрытие турниров; набор ограничен скачанными матчами.")


## Профили команд и игроков
Имена профилей — текущий снимок, а не историческое членство в команде.


In [ ]:
team_profiles = pd.DataFrame([{"team_id": p.get("team_id"), "name": p.get("name"),
    "tag": p.get("tag")} for p in BUNDLE.get("team_profiles", [])])
player_profiles = pd.DataFrame([{"account_id": (p.get("profile") or {}).get("account_id"),
    "name": (p.get("profile") or {}).get("name"),
    "personaname": (p.get("profile") or {}).get("personaname")}
    for p in BUNDLE.get("player_profiles", [])])
display(team_profiles)
display(player_profiles)
print("Профили не подменяют сведения о пятёрках сыгранных матчей.")


## Последние наблюдаемые составы и форма после изменения
A → B → A — три периода, даже если первая пятёрка вернулась.


In [ ]:
latest = teams.sort_values(["started_at", "match_id"]).groupby("team_id").tail(1)
current = latest[latest.roster_known].copy()
current_rows = []
for row in current.itertuples():
    segment = report[(report.team_id == row.team_id) & (report.roster_spell == row.roster_spell)]
    current_rows.append({"team_id":row.team_id, "team":row.team_name,
        "five":row.player_ids, "first_observed":row.roster_first_seen,
        "last_observed":row.started_at, "maps_in_period":len(segment),
        "wins":int(segment.won.sum()),
        "win_rate":round(segment.won.mean()*100,1) if len(segment) else np.nan})
current_rosters = pd.DataFrame(current_rows)
display(current_rosters)
print("Команд с неизвестной пятёркой последней карты:", int((~latest.roster_known).sum()))
print("Пятёрка последней карты может отличаться от объявленного состава будущего матча.")


## Общие результаты и динамика
Общие результаты показаны отдельно от статистики последнего состава.


In [ ]:
summary = report.groupby(["team_id", "team_name"]).agg(
    maps=("won", "size"), wins=("won", "sum"), win_rate=("won", "mean"))
summary["win_rate"] = (summary.win_rate*100).round(1)
display(summary.sort_values("maps", ascending=False))
if not summary.empty:
    summary.sort_values("maps",ascending=False).head(15).reset_index().plot.bar(
        x="team_name",y="win_rate",legend=False,title="Винрейт в собранной выборке, %")
    plt.tight_layout(); plt.show()
    monthly = report.assign(month=report.started_at.dt.strftime("%Y-%m")).groupby(
        ["month", "team_name"]).won.agg(["size", "mean"])
    display(monthly.rename(columns={"size":"maps", "mean":"win_rate"}))


## Пики и баны последних составов
Бан относится к совершившей его команде. Винрейт героя считается только по пикам.


In [ ]:
if not drafts.empty and not current.empty:
    latest_drafts = drafts.merge(current[["team_id", "roster_spell"]],
        on=["team_id", "roster_spell"], how="inner")
    latest_drafts = latest_drafts[latest_drafts.started_at >= ANALYSIS_START].copy()
    latest_drafts["pick_win"] = latest_drafts.is_pick & latest_drafts.won
    hero_stats = latest_drafts.groupby(["team_id", "team_name", "hero_name"]).agg(
        picks=("is_pick","sum"), actions=("is_pick","size"), pick_wins=("pick_win","sum"))
    hero_stats["bans"] = hero_stats.actions-hero_stats.picks
    hero_stats["pick_win_rate"] = (100*hero_stats.pick_wins/hero_stats.picks.replace(0,np.nan)).round(1)
    display(hero_stats.drop(columns="actions").sort_values("picks",ascending=False))
else:
    print("Нет драфтов с известными пятёрками.")


## Игроки отдельно: опыт после переходов и текущая форма
История игрока сохраняется между командами. GPM/KDA относятся к завершённым картам и не являются самостоятельной оценкой силы игрока: роли отличаются.


In [ ]:
if not players.empty:
    real_players = players[(players.account_id > 0) & (players.account_id < 4294967295)]
    recent = real_players[real_players.started_at >= ANALYSIS_START].sort_values(
        ["started_at", "match_id"]).groupby("account_id").tail(20)
    player_form = recent.groupby("account_id").agg(
        maps=("won","size"), win_rate=("won","mean"),
        mean_gpm=("gold_per_min","mean"), mean_xpm=("xp_per_min","mean"),
        teams_played=("team_id","nunique"))
    player_form["win_rate"] = (100*player_form.win_rate).round(1)
    display(player_form)
else:
    print("Данные участников карт отсутствуют.")


## Как читать выводы

- При новом составе командная статистика начинается заново; история отдельных игроков остаётся.
- Неизвестный Steam ID не заменяется игроком из будущей карты.
- Даже пять тех же игроков могут поменять роли; модель пока не учитывает это.
- Стендины создают отдельные периоды. Официальные трансферы требуют отдельного источника.
- Отсутствующие карты могут скрыть промежуточную смену состава.
- Текущий небольшой snapshot не заменяет многолетнюю историю. Прогнозы и проверка модели —
  в `02_roster_forecast.ipynb`.
